### Task 1 
Install all required libraries: 

In [1]:
%pip install langchain 
%pip install langchain-community 
%pip install chromadb 
%pip install sentence-transformers 
%pip install pypdf 

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Task 2 
Load a sample PDF using PyPDFLoader. 

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(r"C:\Users\sachi\Downloads\Information_Bulletin.pdf")
documents = loader.load()


C:\Users\sachi\AppData\Local\Temp\ipykernel_14392\2291049392.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\sachi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Task 3 
Split the PDF into chunks using RecursiveCharacterTextSplitter. 

In [5]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Split the pdf to chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 33


### Task 4 
Store the chunks in ChromaDB and perform a similarity search for three different questions. 

In [9]:
%pip install langchain-huggingface langchain-chroma sentence-transformers


   ---------------------------------------- 0/2 [langchain-huggingface]
   -------------------- ------------------- 1/2 [langchain-chroma]
   ---------------------------------------- 2/2 [langchain-chroma]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#create embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings( 
    model_name="all-MiniLM-L6-v2" 
) 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3511.42it/s]


In [ ]:
#Store in ChromaDB 
from langchain_chroma import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="chroma_db"
)

In [ ]:
#similarity search
results = db.similarity_search(
    "Who is eligible for CAT 2025?",
    k=2
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

CAT 2025 ELIGIBILITY  
Eligibility   
● The candidate must hold a Bachelor’s Degree, with at least 50% marks or equivalent CGPA [45% 
in the case of candidates belonging to the Scheduled Caste (SC), Scheduled Tribe (ST) and Persons 
with Disability (PwD) categories], awarded by any University or educational institution as 
incorporated by an Act of Parliament or State legislature in India or declared to be deemed as an
--------------------------------------------------
January, 2026. The CAT 2025 score is valid only till December 31, 2026 and will accordingly be 
accessible on the website. Thereafter, no queries related to the issuance of CAT 2025 scorecards will 
be entertained.
--------------------------------------------------


In [14]:
results = db.similarity_search(
    "test centre?",
    k=2
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

from a drop-down menu. After the last date of registration, candidates will be allotted one among the 
five preferred cities subject to availability. In the rare case that a candidate is not allotted any of the 
preferred cities, he/she will be allotted a nearby city. The candidates can download the admit cards 
from November 05, 2025 until November 30, 2025.  
  
 
TEST CENTRES  
CAT will be conducted in centres spread across approximately 170 test cities . Test cities will be
--------------------------------------------------
mentioned in the CAT website and the name of the test centre will be indicated in the Admit Card.  
IIMs reserve the right to add/remove, change or cancel any test centre/city and/or change the test time 
and date at their discretion.  
   
CAT 2025 SCORE  
Candidate's CAT 2025 scorecards will be made accessible on the CAT website. Candidates may also 
be intimated individually by SMS. The CAT results are likely to be declared by the First week of
--------------

### Task 5 
Draw the complete architecture of the PDF Question Answering Chatbot. 



```text
                 +----------------------+
                 |      Upload PDF      |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 | PDF Text Extraction  |
                 |   (PyPDFLoader)      |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 |    Text Chunking     |
                 | (RecursiveCharacter  |
                 |   Text Splitter)     |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 | Sentence Transformer |
                 |    (Embeddings)      |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 |      ChromaDB        |
                 |   (Vector Store)     |
                 +----------------------+
                           ▲
                           │
────────────────────────────────────────────────────────────
                           │
                 +----------------------+
                 |    User Question     |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 | Convert to Embedding |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 |  Similarity Search   |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 | Retrieve Best Chunks |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 |   OpenAI / LLM       |
                 |    Generate Answer   |
                 +----------------------+
                           │
                           ▼
                 +----------------------+
                 |    Final Answer      |
                 +----------------------+
```